# 課程08 - 多代理設計模式


## 設定


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 為什麼選擇多代理系統？

現實世界的任務如行程規劃涉及許多不同種類的專業知識——物流、本地知識、預算等等。一個單一代理試圖處理所有事情很快就會變得難以管理。

多代理系統透過**專業化**解決這個問題：每個代理專注於一個專業領域，產出比通才更高品質的結果。它們也提升了**擴展性**——你可以新增代理（例如，航班專家、餐廳評論家）而無需重寫現有的工作流程。代理們通過結構化的管道組合在一起，將上下文從一個傳遞到下一個。


## 創建專門代理人


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## 建立序列工作流程

`WorkflowBuilder` 讓你將代理串接成有向圖。這裡我們建立一個簡單的兩步驟流程：**TravelPlanner** 擬定行程，然後 **TravelConcierge** 審閱並優化它。


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## 新增更多代理人至工作流程

多代理人模式的最大優點之一是擴充性極佳。以下我們新增一個 **BudgetReviewer** 代理人，負責檢查計劃是否符合旅客的預算，標記可能使費用超出限制的項目，並建議節省開支的替代方案。工作流程現在依序執行三個代理人：

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Summary

在本課程中，您學到了如何：

1. **創建專門化代理** — 每個代理都有專注的角色（規劃、禮賓、預算審查）。
2. **使用 `WorkflowBuilder` 和 `add_edge` 將代理串接成順序工作流程**。
3. **從多代理管線串流輸出**，並追蹤發言的代理。
4. **透過新增代理擴展工作流程**，而不修改現有代理。

多代理設計模式讓每個代理保持簡單，同時產出比單一代理更豐富、經過更完整審核的結果。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：  
本文件係使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們致力於確保翻譯的準確性，但請注意自動翻譯可能包含錯誤或不精確之處。原始文件之母語版本應視為權威來源。對於重要資訊，建議採用專業人工翻譯。我們不對因使用此翻譯所引起的任何誤解或曲解承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
